# 01 — Tải dữ liệu Junyi Academy

**Mục đích:** tải bộ dữ liệu Junyi Academy (bản 01/2015) về Google Drive, kiểm tra toàn vẹn và giải nén.

- **Nguồn gốc:** PSLC DataShop, `datasetId=1198`. Bản sao tải tự do trên máy chủ dữ liệu USTC (thư viện EduData sử dụng).
- **Điều kiện sử dụng:** không dùng cho mục đích thương mại; trích dẫn Chang, Hsu & Chen (2015), *Modeling Exercise Relationships in E-Learning: A Unified Approach*, EDM 2015.
- **Dung lượng:** tệp nén 1,3 GB, sau giải nén thêm 2,7 GB (chỉ lấy nhật ký gốc, bỏ qua tệp định dạng PSLC 9,7 GB). Cần khoảng **4 GB** trống trên Drive.
- **Thời gian:** chỉ cần chạy **một lần**. Nếu phiên Colab bị ngắt khi đang tải, chạy lại notebook, quá trình tải sẽ tiếp tục từ chỗ dừng.

Không cần GPU cho notebook này.

In [ ]:
# Gắn Google Drive, lấy mã nguồn mới nhất của kho GraphK và cài gói graphk
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE = '/content/drive/MyDrive/GraphK'
REPO = '/content/GraphK'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Trilosphan/GraphK.git {REPO}
else:
    # đồng bộ đúng với GitHub (bỏ mọi thay đổi trong bản sao trên Colab)
    !git -C {REPO} fetch -q && git -C {REPO} reset -q --hard origin/main
%pip install -q -e {REPO}
sys.path.insert(0, f'{REPO}/src')
!git -C {REPO} log --oneline -1

## 1. Công cụ giải nén

In [ ]:
import shutil
if not shutil.which('unrar'):
    !apt-get -qq install -y unrar > /dev/null
print('unrar:', shutil.which('unrar'))

## 2. Tải tệp nén và kiểm tra toàn vẹn

In [ ]:
from pathlib import Path
from graphk.data import download_junyi, sha256, JUNYI_SHA256

RAW = Path(DRIVE) / 'raw'
archive = download_junyi(RAW)
digest = sha256(archive)
(RAW / 'SHA256SUMS').write_text(f'{digest}  {archive.name}\n')
print(archive, archive.stat().st_size, 'byte')
print('SHA256:', digest)
assert digest == JUNYI_SHA256, 'SHA256 khác giá trị đã ghi nhận — tệp tải về có thể hỏng hoặc máy chủ đã thay tệp'

## 3. Giải nén

In [ ]:
from graphk.data import extract_junyi

paths = extract_junyi(archive, RAW / 'junyi')
for k, v in paths.items():
    print(f'{k:15s} {v}  ({v.stat().st_size / 1e9:.2f} GB)')

## 4. Xem nhanh bảng bài tập

In [ ]:
import pandas as pd

table = pd.read_csv(paths['exercise_table'])
print('Số dòng (bài tập):', len(table))
print('Số bài còn hoạt động (live):', int(table['live'].astype(str).str.upper().eq('TRUE').sum()))
print('Số bài không có tiên quyết:', int(table['prerequisites'].isna().sum()))
table.head()